# C-02: Entity facts shouldn't be from conflicting organisations

**Author**: Sian Wilson <br>
**Date created**: 24th November 2026 <br>
**Dataset Scope**: all datasets on the platform with a published Datasette database and at least two active local authorities or LPAs supplying it <br>
**Purpose**: Ticket C-02 ([digital-land/config#2992](https://github.com/digital-land/config/issues/2992)) — find entities with facts from two or more *active* organisations, and work out why. <br>

**Summary**

- **119 of 816,287 entities** have facts from two active LPAs: 112 in `brownfield-land` and 7 in `conservation-area`. None of the other 20 datasets checked have any. The other 101 published datasets have fewer than two active local authorities or LPAs supplying them, so a conflict isn't possible there.
- **`brownfield-land`: two files registered to the wrong council.** A Mid Sussex file is registered to Mid Suffolk (64 entities, since December 2020) and a Broads Authority file to Broadland (48, since January 2025). Both sources are still open.
- **`conservation-area`: redirects between two LPAs.** An entity mapped to one active LPA was redirected into another's. All 7 redirects were made between November 2025 and October 2026, so this is still happening.
- **382 more** are shared between an LPA and the GLA or Liverpool City Region. These come from old regional registers, none still current.
- **Next steps:** correct the two sources, review the 7 redirects (starting with Thame and Henley), and add a check that stops a redirect joining two active LPAs.

**How it works**

- An organisation is in scope if it is an LPA (has a `local_planning_authority` code, which includes national parks and some development corporations) or has the type `local-authority` (which adds counties, combined authorities and the GLA).
- An organisation with an `end_date` is ignored, so facts from a council that has been merged into a new one never count.
- Facts are traced to organisations by `fact_resource` → `resource_organisation`. `fact_resource` has already had redirects applied, so two LPAs' entities that were redirected into one are included.
- Any field counts, and so do old facts, as long as the organisation is still active.
- A file registered to several active organisations can't show which of them supplied a fact, so it is left out.
- Government bodies (e.g. Historic England, MHCLG) are out of scope, as their national compilations are expected to overlap with LPA data.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

from helpers import chunk, datasette_sql, fetch_config_csv, fetch_table_csv, list_published_datasets

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = os.path.join("..", "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)


def in_list(values):
    return ",".join(f"'{v}'" for v in values)

## 1. Active organisations and their files

In [2]:
orgs = fetch_table_csv("organisation")
orgs["is_lpa"] = orgs["local_planning_authority"].notna()
in_scope = orgs[orgs["is_lpa"] | (orgs["dataset"] == "local-authority")]
active = in_scope[in_scope["end_date"].isna()]
print(f"{len(in_scope)} organisations in scope: {len(active)} active ({active['is_lpa'].sum()} LPAs), {len(in_scope) - len(active)} ended and ignored")

resource_dataset = fetch_table_csv("resource_dataset")
resource_org = fetch_table_csv("resource_organisation")[["resource", "organisation"]]
resource_org["organisation"] = resource_org["organisation"].str.replace("-eng", "", regex=False)
resource_org = resource_org[resource_org["organisation"].isin(active["organisation"])].drop_duplicates()
resource_org["shared"] = resource_org.groupby("resource")["organisation"].transform("nunique") > 1

394 organisations in scope: 343 active (310 LPAs), 51 ended and ignored


In [3]:
def resources_for_dataset(dataset):
    """Resources with facts in a dataset. The biggest tables hit the SQL time limit, so use resource_dataset for those."""
    try:
        return dataset, set(datasette_sql(dataset, "SELECT DISTINCT resource FROM fact_resource ORDER BY resource")["resource"]), "fact_resource"
    except ValueError as e:
        if "no such table" in str(e):
            return dataset, set(), "no fact_resource table"
        return dataset, set(resource_dataset.loc[resource_dataset["dataset"] == dataset, "resource"]), "resource_dataset"


with ThreadPoolExecutor(max_workers=8) as pool:
    found = list(pool.map(resources_for_dataset, sorted(list_published_datasets())))

coverage, files = [], []
for dataset, resources, source in found:
    d = resource_org[resource_org["resource"].isin(resources)]
    usable = d[~d["shared"]]
    files.append(usable.assign(dataset=dataset))
    coverage.append({"dataset": dataset, "resource list from": source, "organisations": usable["organisation"].nunique(),
                     "files": usable["resource"].nunique(), "shared files left out": d.loc[d["shared"], "resource"].nunique()})

coverage = pd.DataFrame(coverage).set_index("dataset")
no_facts = coverage.index[coverage["resource list from"] == "no fact_resource table"].tolist()
files = pd.concat(files)
files = files[files["dataset"].isin(coverage.index[coverage["organisations"] > 1])]
print(f"{len(coverage)} published datasets, {files['dataset'].nunique()} with two or more active organisations. No fact_resource table: {no_facts}")
coverage[coverage["organisations"] > 1].sort_values("files", ascending=False)

123 published datasets, 22 with two or more active organisations. No fact_resource table: ['title-boundary']


,resource list from,organisations,files,shared files left out
dataset,,,,
article-4-direction-area,fact_resource,118,3378,0
listed-building-outline,resource_dataset,104,3157,5
tree-preservation-zone,resource_dataset,86,3021,6
tree,resource_dataset,80,1949,5
brownfield-land,fact_resource,302,1895,28
conservation-area,fact_resource,147,1675,1
tree-preservation-order,resource_dataset,80,1269,3
conservation-area-document,fact_resource,97,1140,0
brownfield-site,fact_resource,4,962,0


## 2. Which entities does each organisation have facts for?

One query per dataset and organisation, in batches of 40 files. If a batch hits the SQL time limit it is split in half. Takes around 5 minutes.

In [4]:
jobs = [(dataset, org, batch) for (dataset, org), g in files.groupby(["dataset", "organisation"]) for batch in chunk(sorted(g["resource"]), 40)]
failed = []


def entities_for_files(dataset, resources):
    try:
        e = datasette_sql(dataset, f"SELECT GROUP_CONCAT(DISTINCT entity) AS e FROM fact_resource WHERE resource IN ({in_list(resources)})")["e"][0]
    except Exception as err:
        if len(resources) == 1:
            failed.append((dataset, resources[0], str(err)[:150]))
            return set()
        half = len(resources) // 2
        return entities_for_files(dataset, resources[:half]) | entities_for_files(dataset, resources[half:])
    return {int(i) for i in e.split(",")} if e else set()


with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(lambda j: entities_for_files(j[0], j[2]), jobs))

entity_orgs = pd.DataFrame([(ds, org, e) for (ds, org, _), ents in zip(jobs, results) for e in ents],
                           columns=["dataset", "organisation", "entity"]).drop_duplicates()
entity_orgs = entity_orgs.merge(orgs[["organisation", "name", "is_lpa"]], on="organisation")
print(f"{len(jobs)} batches, {len(failed)} files failed {failed}")
print(f"{len(entity_orgs[['dataset', 'entity']].drop_duplicates()):,} entities checked")

1879 batches, 0 files failed []
816,287 entities checked


## 3. Entities with facts from two or more active organisations

In [5]:
g = entity_orgs.sort_values("name").groupby(["dataset", "entity"])
conflicts = g.agg(n_orgs=("organisation", "nunique"), n_lpas=("is_lpa", "sum"), organisations=("name", " | ".join)).reset_index()
conflicts = conflicts[conflicts["n_orgs"] > 1]
conflicts["conflict"] = (conflicts["n_lpas"] > 1).map({True: "two active LPAs", False: "LPA and another local authority"})

pairs = conflicts.groupby(["conflict", "dataset", "organisations"]).size().rename("entities").reset_index()
pairs = pairs.sort_values(["conflict", "entities"], ascending=False)
print(conflicts.groupby("conflict").size().to_string())
pairs

conflict
LPA and another local authority    382
two active LPAs                    119


,conflict,dataset,organisations,entities
5,two active LPAs,brownfield-land,Mid Suffolk District Council | Mid Sussex District Council,64
4,two active LPAs,brownfield-land,Broadland District Council | Broads Authority,48
9,two active LPAs,conservation-area,New Forest District Council | New Forest National Park Authority,3
6,two active LPAs,conservation-area,Buckinghamshire Council | South Oxfordshire District Council,2
7,two active LPAs,conservation-area,City of Bradford Metropolitan District Council | Leeds City Council,1
8,two active LPAs,conservation-area,Huntingdonshire District Council | Peterborough City Council,1
1,LPA and another local authority,brownfield-site,Greater London Authority | London Borough of Southwark,346
0,LPA and another local authority,brownfield-land,Halton Borough Council | Liverpool City Region,34
2,LPA and another local authority,central-activities-zone,Greater London Authority | London Borough of Lambeth,1
3,LPA and another local authority,central-activities-zone,Greater London Authority | London Borough of Southwark,1


## 4. Why?

For each pairing: the organisation the entity is currently assigned to, who the files are registered to, where they were downloaded from, and whether an older entity was redirected into it.

In [6]:
trace, details = [], []
for dataset, c in conflicts.groupby("dataset"):
    # via fact, as fact_resource has no index on entity
    fr = pd.concat([datasette_sql(dataset, f"""
            SELECT f.entity, fr.resource FROM fact f JOIN fact_resource fr ON fr.fact = f.fact
            WHERE f.entity IN ({','.join(map(str, b))}) GROUP BY f.entity, fr.resource ORDER BY f.entity, fr.resource""")
        for b in chunk(c["entity"], 20)])
    fr = fr.merge(files.loc[files["dataset"] == dataset, ["resource", "organisation"]], on="resource")
    ep = pd.concat([datasette_sql("digital-land", f"""
            SELECT re.resource, r.start_date AS file_start, r.end_date AS file_end, e.endpoint_url FROM resource_endpoint re
            JOIN resource r ON r.resource = re.resource JOIN endpoint e ON e.endpoint = re.endpoint
            WHERE re.resource IN ({in_list(b)}) ORDER BY re.resource, e.endpoint""")
        for b in chunk(sorted(fr["resource"].unique()), 50)])
    trace.append(fr.merge(ep, on="resource").assign(dataset=dataset))

    ent = pd.concat([datasette_sql(dataset, f"SELECT entity, name, reference, organisation_entity FROM entity WHERE entity IN ({','.join(map(str, b))}) ORDER BY entity")
                     for b in chunk(c["entity"], 20)])
    redirects = fetch_config_csv(dataset, "old-entity")
    ent["redirected_from"] = ent["entity"].map(redirects.groupby("entity")["old-entity"].apply(list)) if len(redirects) else None
    details.append(ent.assign(dataset=dataset))

details = pd.concat(details)
details["current_organisation"] = pd.to_numeric(details["organisation_entity"]).map(orgs.set_index("entity")["name"])
out = conflicts.merge(details.drop(columns="organisation_entity"), on=["dataset", "entity"], how="left")
out["entity_url"] = "https://www.planning.data.gov.uk/entity/" + out["entity"].astype(str)
trace = pd.concat(trace).merge(out[["dataset", "entity", "organisations", "conflict"]], on=["dataset", "entity"])
trace["organisation"] = trace["organisation"].map(orgs.set_index("organisation")["name"])
trace["current file"] = trace["file_end"].isna() | (trace["file_end"] == "")

(out.assign(redirected=out["redirected_from"].notna())
 .groupby(["conflict", "dataset", "organisations", "current_organisation"], dropna=False)
 .agg(entities=("entity", "size"), redirected=("redirected", "sum")).reset_index()
 .sort_values(["conflict", "entities"], ascending=False))

,conflict,dataset,organisations,current_organisation,entities,redirected
5,two active LPAs,brownfield-land,Mid Suffolk District Council | Mid Sussex District Council,Mid Sussex District Council,64,0
4,two active LPAs,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,48,0
6,two active LPAs,conservation-area,Buckinghamshire Council | South Oxfordshire District Council,Buckinghamshire Council,2,2
10,two active LPAs,conservation-area,New Forest District Council | New Forest National Park Authority,New Forest National Park Authority,2,2
7,two active LPAs,conservation-area,City of Bradford Metropolitan District Council | Leeds City Council,Leeds City Council,1,1
8,two active LPAs,conservation-area,Huntingdonshire District Council | Peterborough City Council,Huntingdonshire District Council,1,1
9,two active LPAs,conservation-area,New Forest District Council | New Forest National Park Authority,New Forest District Council,1,1
1,LPA and another local authority,brownfield-site,Greater London Authority | London Borough of Southwark,London Borough of Southwark,346,0
0,LPA and another local authority,brownfield-land,Halton Borough Council | Liverpool City Region,Halton Borough Council,34,0
2,LPA and another local authority,central-activities-zone,Greater London Authority | London Borough of Lambeth,London Borough of Lambeth,1,0


In [7]:
urls = trace.groupby(["dataset", "organisations", "organisation", "endpoint_url"]).agg(
    entities=("entity", "nunique"), first_file=("file_start", "min"), has_current_file=("current file", "any")).reset_index()
with pd.option_context("display.max_colwidth", None):
    display(urls[urls["dataset"] != "brownfield-site"])

,dataset,organisations,organisation,endpoint_url,entities,first_file,has_current_file
0,brownfield-land,Broadland District Council | Broads Authority,Broadland District Council,https://www.broads-authority.gov.uk/__data/assets/text_file/0025/538027/broads_authority_brownfield_register_2024_11_18.csv,48,2025-01-07,True
1,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,http://www.broads-authority.gov.uk/__data/assets/text_file/0004/1064128/broads_authority_brownfieldregister_2017-12-13_rev1.csv,8,2018-05-22,False
2,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,http://www.broads-authority.gov.uk/__data/assets/text_file/0018/1432071/broads_authority_brownfieldregister_2018-12-20_rev1.csv,16,2019-12-01,False
3,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,https://www.broads-authority.gov.uk/__data/assets/text_file/0018/1432071/broads_authority_brownfieldregister_2019_12_04.csv,25,2019-12-14,False
4,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,https://www.broads-authority.gov.uk/__data/assets/text_file/0028/256681/broads_authority_brownfieldregister_2019_12_04.csv,25,2019-12-14,False
5,brownfield-land,Broadland District Council | Broads Authority,Broads Authority,https://www.broads-authority.gov.uk/brownfield_register,48,2020-12-21,True
6,brownfield-land,Halton Borough Council | Liverpool City Region,Halton Borough Council,http://inspire.halton.gov.uk/data/halton_brownfieldregister_2017-12-31_rev1.csv,30,2018-05-22,False
7,brownfield-land,Halton Borough Council | Liverpool City Region,Halton Borough Council,http://inspire.halton.gov.uk/data/halton_brownfieldregister_2018-11-06_rev1.csv,34,2019-12-14,False
8,brownfield-land,Halton Borough Council | Liverpool City Region,Halton Borough Council,https://www4.halton.gov.uk/Documents/planning/brownfieldregisters/brownfield-sites.csv,23,2024-01-18,False
9,brownfield-land,Halton Borough Council | Liverpool City Region,Liverpool City Region,https://www.liverpoolcityregion-ca.gov.uk/wp-content/uploads/LCRCA-Brownfield-Land-Register-2019-Updated.xlsx,34,2020-03-10,False


### Were the conservation areas redirected from another LPA's entity?

`lookup.csv` shows which organisations were mapped to the old entity and to the one it was redirected to.

In [8]:
lookup = fetch_config_csv("conservation-area", "lookup")
old = fetch_config_csv("conservation-area", "old-entity")
active_lpas = active[active["is_lpa"]].set_index("organisation")["name"]
lookup["lpa"] = lookup["organisation"].map(active_lpas)
mapped = lookup.dropna(subset=["lpa"]).groupby("entity")["lpa"].agg(lambda x: " | ".join(sorted(set(x))))

ca = out[out["dataset"] == "conservation-area"].merge(old[["old-entity", "entity", "entry-date"]], on="entity")
ca["old entity mapped to"] = ca["old-entity"].map(mapped)
ca["entity mapped to"] = ca["entity"].map(mapped)
ca[["entity", "name", "entity mapped to", "old-entity", "old entity mapped to", "entry-date"]].rename(columns={"entry-date": "redirected"})

,entity,name,entity mapped to,old-entity,old entity mapped to,redirected
0,44010474,Thame,Buckinghamshire Council,44004364,South Oxfordshire District Council,2025-11-10
1,44010475,"Henley, Main",Buckinghamshire Council,44004415,South Oxfordshire District Council,2026-02-06
2,44014906,RAWDON - LITTLE LONDON,Leeds City Council,44004991,City of Bradford Metropolitan District Council,2026-02-11
3,44017415,Buckland,New Forest District Council,44006961,New Forest National Park Authority,2026-10-23
4,44017420,Ashlett Creek,New Forest National Park Authority,44007335,New Forest District Council,2026-10-26
5,44017421,Keyhaven,New Forest National Park Authority,44005388,New Forest District Council,2026-10-26
6,44017423,ALWALTON,Huntingdonshire District Council,44004541,Peterborough City Council,2026-10-26


**Two causes:**

1. **A file registered to the wrong organisation** (`brownfield-land`). `midsussex.gov.uk/media/5664` is registered to `local-authority:MSU` (Mid Suffolk) instead of `MSS`, and `broads_authority_brownfield_register_2024_11_18.csv` to `local-authority:BRO` (Broadland) instead of `national-park-authority:Q4972284`. None of these entities were redirected.
2. **Redirects between two active LPAs** (`conservation-area`). Every one had an older entity mapped to one LPA redirected into the other's. Some look like areas on or near a shared boundary that both LPAs publish, so keeping one entity may be right, but the assigned organisation should be checked.

**LPA and another local authority:** the GLA and Liverpool City Region published registers covering several councils. These may be deliberate, so they aren't counted as errors.

**What this can't see:** files registered to several active organisations are left out. Most look like genuine joint services (e.g. Adur and Worthing), but one `brownfield-land` file is registered to both Wyre and Wyre Forest, which looks like another mix-up.

## 5. Save

In [9]:
out.to_csv(os.path.join(DATA_DIR, "C-02_conflicting_organisations.csv"), index=False)
print(f"{len(out)} entities saved")

501 entities saved
